# Speed: the fast Grassmann layer

The paper projects each offset's 496 Plücker numbers to 256 and then averages the six results. Every pair shares the current token z_t, and the Plücker map is linear in its second vector, so the average equals **one** pair: z_t with a weighted average of its past window (the weights include the normaliser, from Lagrange's identity ‖a∧b‖² = ‖a‖²‖b‖² − (a·b)²). The fast path builds that one pair per token and folds the upper-triangle selection into the projection weights, so there is no slow per-token lookup. Same maths, far fewer operations.

1. **Same numbers?** Fast path vs the paper's order on this GPU, in float32: outputs and every gradient.
2. **How much faster?** Training tokens/s for each model, fast vs paper order.
3. **Where does the time go?** A profile of a few training steps.
4. **Are matrix multiplies using tensor cores?** Plain matmul speed at this model's shapes, by precision and BLAS library. On the L4 the profile showed every matmul on a slow fallback kernel (`magma_sgemmEx`); this finds which setting fixes it.

No data download is needed: speed is measured on random tokens. About 15 minutes on an A100 (compiling each model takes most of it).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, json, subprocess, sys
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/speed.json'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers
# drop any grassman modules imported earlier in this runtime, so the code just pulled is what runs
for _name in [k for k in sys.modules if k == 'grassman' or k.startswith('grassman.')]:
    del sys.modules[_name]

Only if you are **resuming this notebook** in a new runtime: set `RESTORE = True` to upload the `results.zip` it downloaded earlier, so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

## 1. Same numbers?

Differences of about 1e-5 or less are float32 rounding; anything larger is a bug.

In [ ]:
import torch
from grassman.bench import fast_vs_reference, throughput, set_reference
dev = torch.device('cuda')
for r in fast_vs_reference(dev):
    print(f"{r['kind']:<8} normalize={str(r['normalize']):<5}  max output diff {r['max_out_diff']:.2e}   "
          f"max gradient diff {r['max_grad_diff']:.2e}")

## 2. How much faster? (training tokens/s, batch 32 × 128)

Precision follows the training rule: bf16 on A100/L4/H100, fp16 with loss scaling on T4. Each model is timed eager (normal PyTorch) and compiled (`torch.compile`, which fuses small operations); the paper's order is timed eager for the pair models. Compiling takes a while per model (shown in brackets). About 10–15 minutes in total.

In [ ]:
from grassman.model import Config
from grassman.train import pick_amp
print('precision on this GPU:', pick_amp(dev))
results = {}
PAPER = ('grassmann', 'sym', 'outer', 'concat', 'concat32', 'g-dilated')
for m in ['grassmann', 'sym', 'outer', 'concat', 'concat32', 'g-select', 'g-dilated', 'g-both', 'g-both-angle',
          'hyena', 'swa', 'attention']:
    cfg = Config(mixer=m)
    eager = throughput(cfg, dev)
    comp = throughput(cfg, dev, compile=True)
    ref = throughput(cfg, dev, reference=True) if m in PAPER else None
    results[m] = {'eager': eager, 'compiled': comp, 'paper_order': ref}
    line = (f"{m:<13} eager {eager['tok_per_s']:>9,.0f}   compiled {comp['tok_per_s']:>9,.0f} "
            f"({comp['warmup_s']:3.0f}s)")
    if ref:
        line += (f"   paper order {ref['tok_per_s']:>9,.0f}   vs paper: eager {eager['tok_per_s'] / ref['tok_per_s']:.1f}x, "
                 f"compiled {comp['tok_per_s'] / ref['tok_per_s']:.1f}x")
    print(line, flush=True)
json.dump({'gpu': torch.cuda.get_device_name(), 'results': results}, open(RUNS, 'w'), indent=1)

## 3. Where does the time go?

Top GPU operations over 10 training steps. If the same operations dominate for `attention` and `grassmann`, the cost is in the shared parts (embedding, output layer, loss, optimizer), not the mixer.

In [ ]:
from torch.profiler import profile, ProfilerActivity
from grassman.model import LM
for m in ['attention', 'grassmann']:
    torch.manual_seed(0)
    model = LM(Config(mixer=m)).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.randint(0, 30522, (32, 128), device=dev)
    amp = pick_amp(dev)
    def step():
        with torch.autocast('cuda', dtype=amp):
            _, loss = model(x, x)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    for _ in range(5):
        step()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(10):
            step()
        torch.cuda.synchronize()
    print(f'===== {m} =====')
    print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))

## 4. Are matrix multiplies using tensor cores?

Time of one matmul at this model's shapes. For scale, peak dense throughput is roughly 120 TFLOP/s (bf16/fp16) and 30 (fp32) on an L4, and 312 and 19.5 on an A100. A bf16 number near the fp32 one, or far below it, means the tensor cores are not being used. The vocabulary (30,522) is not a multiple of 8, which can block the fast kernels, so a padded version (30,528) is timed too. The clock and power line shows whether the GPU is throttling.

In [ ]:
!nvidia-smi --query-gpu=name,clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu --format=csv
import time
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name())
try:
    default_blas = torch.backends.cuda.preferred_blas_library()
    print('preferred BLAS library:', default_blas)
except Exception as e:
    default_blas = None
    print('preferred_blas_library not available:', e)

def gemm(M, K, N, dtype, iters=20):
    a = torch.randn(M, K, device=dev, dtype=dtype)
    b = torch.randn(K, N, device=dev, dtype=dtype)
    for _ in range(3):
        a @ b
    torch.cuda.synchronize(); t = time.time()
    for _ in range(iters):
        a @ b
    torch.cuda.synchronize()
    ms = (time.time() - t) / iters * 1e3
    return ms, 2 * M * K * N / (ms / 1e3) / 1e12

SHAPES = {'output layer  4096x256 @ 256x30522': (4096, 256, 30522),
          'padded vocab  4096x256 @ 256x30528': (4096, 256, 30528),
          'FFN           4096x256 @ 256x1024': (4096, 256, 1024)}
PRECISIONS = [('bf16', torch.bfloat16, False), ('fp16', torch.float16, False),
              ('fp32', torch.float32, False), ('fp32+TF32', torch.float32, True)]
for lib in ['cublas', 'cublaslt']:
    try:
        torch.backends.cuda.preferred_blas_library(lib)
    except Exception as e:
        print(f'{lib}: cannot select ({e})'); continue
    for pname, dtype, tf32 in PRECISIONS:
        torch.backends.cuda.matmul.allow_tf32 = tf32
        for sname, (M, K, N) in SHAPES.items():
            ms, tflops = gemm(M, K, N, dtype)
            print(f'{lib:<9} {pname:<10} {sname}   {ms:7.3f} ms   {tflops:6.1f} TFLOP/s')
    print()
torch.backends.cuda.matmul.allow_tf32 = False
if default_blas is not None:
    torch.backends.cuda.preferred_blas_library(default_blas)

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')